# Conic10K real data audit (Colab)
Downloads the real Conic10K dataset, audits it, creates a fixed 100-item review sample, and zips outputs.


In [ ]:
!python -m pip install -q datasets pandas pyarrow


In [ ]:
from pathlib import Path
import json, csv, random, html, zipfile
from collections import Counter
from datasets import load_dataset

DATASET_ID='WenyangHui/Conic10K'
SOURCE_URL='https://huggingface.co/datasets/WenyangHui/Conic10K'
LICENSE='MIT'
OUT=Path('/content/conic10k_audit_outputs'); OUT.mkdir(exist_ok=True)
print({'source': SOURCE_URL, 'dataset_version': DATASET_ID, 'license': LICENSE})
ds=load_dataset(DATASET_ID)
print(ds)


In [ ]:
TEXT_KEYS=('text','question','problem','input')
ANSWER_KEYS=('answer','output','target','solution')
def pick(row, keys):
    for k in keys:
        if k in row and row[k] is not None: return str(row[k])
    return ''
rows=[]; split_counts=Counter(); missing=Counter(); lengths=[]; seen={}; dupes=[]
for split in ds.keys():
    for idx,row in enumerate(ds[split]):
        row=dict(row); text=pick(row,TEXT_KEYS); answer=pick(row,ANSWER_KEYS); formal=str(row.get('formal', row.get('program', row.get('representation',''))) or '')
        item_id=str(row.get('id', f'{split}-{idx}'))
        split_counts[split]+=1; lengths.append(len(text))
        for name,val in [('text',text),('answer',answer),('formal',formal)]:
            if not val: missing[name]+=1
        key=(text,answer,formal)
        if key in seen: dupes.append((seen[key], item_id))
        else: seen[key]=item_id
        rows.append({'id':item_id,'split':split,'text':text,'answer':answer,'formal':formal})
sl=sorted(lengths)
def pct(p): return sl[min(len(sl)-1, round((len(sl)-1)*p))] if sl else 0
stats={'source_url':SOURCE_URL,'dataset_version':DATASET_ID,'license':LICENSE,'total_items':len(rows),'split_counts':dict(split_counts),'missing_counts':dict(missing),'duplicate_count':len(dupes),'text_length':{'min':min(sl),'p50':pct(.5),'p95':pct(.95),'max':max(sl)}}
print(json.dumps(stats, ensure_ascii=False, indent=2))
(OUT/'audit_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding='utf-8')
report=f'''# Conic10K Data Audit Report

STATUS: RUN ON REAL CONIC10K

- Source: {SOURCE_URL}
- Dataset/version: {DATASET_ID}
- License: {LICENSE}
- Total items: {len(rows)}
- Split counts: {json.dumps(dict(split_counts), ensure_ascii=False)}
- Missing counts: {json.dumps(dict(missing), ensure_ascii=False)}
- Duplicate count: {len(dupes)}
- Text length: {json.dumps(stats['text_length'], ensure_ascii=False)}
'''
(OUT/'DATA_AUDIT_REPORT.md').write_text(report, encoding='utf-8')


In [ ]:
sample=random.Random(20240623).sample(rows,100)
fields=['sample_index','id','split','text','answer','formal','review_keep','review_issue','review_notes']
csv_path=OUT/'conic10k_100_item_review_sample.csv'; jsonl_path=OUT/'conic10k_100_item_review_sample.jsonl'; html_path=OUT/'conic10k_100_item_review_sample.html'
with csv_path.open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fields); w.writeheader()
    for i,r in enumerate(sample,1): w.writerow({'sample_index':i, **r, 'review_keep':'','review_issue':'','review_notes':''})
with jsonl_path.open('w',encoding='utf-8') as f:
    for i,r in enumerate(sample,1): f.write(json.dumps({'sample_index':i, **r, 'review_keep':'','review_issue':'','review_notes':''}, ensure_ascii=False)+'\\n')
trs=[]
for i,r in enumerate(sample,1): trs.append('<tr>'+''.join(f'<td>{html.escape(str(v))}</td>' for v in [i,r['id'],r['split'],r['text'],r['answer'],r['formal'],'','',''])+'</tr>')
html_path.write_text('<html><meta charset="utf-8"><body><table border="1"><tr>'+''.join(f'<th>{c}</th>' for c in fields)+'</tr>'+''.join(trs)+'</table></body></html>', encoding='utf-8')
zip_path=Path('/content/conic10k_audit_outputs.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir(): z.write(p,p.name)
print('Created:', zip_path)
print('Total items:', len(rows), 'Split counts:', dict(split_counts))


In [ ]:
from google.colab import files
files.download('/content/conic10k_audit_outputs.zip')
